# GSM8K dataset exploration

Pure EDA on the local `data/gsm8k/*.jsonl` files — no model, no prompts, no grading.
Stdlib only, so it runs in any kernel and downloads nothing.

Goal: get a feel for the data you'll evaluate `OLMo-2-0425-1B` on for `prompting_baselines`.

In [1]:
import json
import statistics
from collections import Counter
from pathlib import Path

DATA_DIR = Path("data/gsm8k")

def load_jsonl(path):
    """Each line is an independent JSON object: {'question': str, 'answer': str}."""
    with open(path) as f:
        return [json.loads(line) for line in f]

test = load_jsonl(DATA_DIR / "test.jsonl")
train = load_jsonl(DATA_DIR / "train.jsonl")
print(f"test:  {len(test):>5} examples")
print(f"train: {len(train):>5} examples")

test:   1319 examples
train:  7473 examples


## 1. Look at a few raw records

The `answer` field holds the full chain-of-thought rationale, with the final numeric answer after `####`.

In [2]:
for ex in test[:2]:
    print("Q:", ex["question"])
    print("-" * 80)
    print("A:", ex["answer"])
    print("=" * 80)

Q: Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?
--------------------------------------------------------------------------------
A: Janet sells 16 - 3 - 4 = <<16-3-4=9>>9 duck eggs a day.
She makes 9 * 2 = $<<9*2=18>>18 every day at the farmer’s market.
#### 18
Q: A robe takes 2 bolts of blue fiber and half that much white fiber.  How many bolts in total does it take?
--------------------------------------------------------------------------------
A: It takes 2/2=<<2/2=1>>1 bolt of white fiber
So the total amount of fabric is 2+1=<<2+1=3>>3 bolts of fabric
#### 3


In [10]:
test[0]['question']

"Janet’s ducks lay 16 eggs per day. She eats three for breakfast every morning and bakes muffins for her friends every day with four. She sells the remainder at the farmers' market daily for $2 per fresh duck egg. How much in dollars does she make every day at the farmers' market?"

## 2. Extract the gold final answer (split on `####`)

This is the `ground_truth` you'd hand to the reward functions — the stripped value after `####`,
**not** the whole rationale.

In [4]:
def gold_answer(record):
    return record["answer"].split("####")[-1].strip()

for ex in test[:5]:
    print(repr(gold_answer(ex)))

'18'
'3'
'70000'
'540'
'20'


## 3. How long are questions and rationales?

Useful for sanity-checking your generation `max_tokens` (the handout uses 512).

In [5]:
def summarize(name, values):
    values = sorted(values)
    n = len(values)
    pct = lambda p: values[min(n - 1, int(p / 100 * n))]
    print(f"{name:>22}:  min={values[0]:>4}  median={statistics.median(values):>6.0f}  "
          f"p90={pct(90):>4}  p99={pct(99):>4}  max={values[-1]:>4}")

q_words = [len(ex["question"].split()) for ex in test]
a_words = [len(ex["answer"].split()) for ex in test]
# number of reasoning steps ~ number of newline-separated lines in the rationale
steps   = [ex["answer"].count("\n") for ex in test]

summarize("question words", q_words)
summarize("rationale words", a_words)
summarize("reasoning steps (\\n)", steps)

        question words:  min=  15  median=    43  p90=  70  p99= 102  max= 164
       rationale words:  min=   5  median=    49  p90=  89  p99= 141  max= 173
  reasoning steps (\n):  min=   2  median=     3  p90=   6  p99=   8  max=  11


In [6]:
# Tiny text histogram of reasoning-step counts
hist = Counter(steps)
for k in sorted(hist):
    print(f"{k:>2} steps | {'#' * (hist[k] * 60 // max(hist.values()))} {hist[k]}")

 2 steps | #################################################### 326
 3 steps | ############################################################ 370
 4 steps | ################################################ 298
 5 steps | ############################ 174
 6 steps | ############## 88
 7 steps | ###### 40
 8 steps | ### 20
 9 steps |  2
11 steps |  1


## 4. What do the gold answers look like?

Are they all clean integers? Any decimals, negatives, commas, or units lurking?
(This matters because the grader normalizes strings before comparing.)

In [7]:
golds = [gold_answer(ex) for ex in test]

def classify(ans):
    s = ans.replace(",", "")
    try:
        int(s); return "int"
    except ValueError:
        pass
    try:
        float(s); return "float"
    except ValueError:
        return "non-numeric"

print(Counter(classify(g) for g in golds))
print("\nAny gold answers containing a comma:")
print([g for g in golds if "," in g][:10])
print("\nAny non-numeric gold answers:")
print([g for g in golds if classify(g) == "non-numeric"][:10])

Counter({'int': 1319})

Any gold answers containing a comma:
['2,125', '114,200', '276,000', '5,600', '1,600', '65,960', '1,450,000', '43,500', '10,800', '6,250']

Any non-numeric gold answers:
[]


## 5. Scratch cell

Poke at individual examples — e.g. find the longest question, or the example with the most steps.

In [8]:
hardest = max(test, key=lambda ex: ex["answer"].count("\n"))
print("Q:", hardest["question"])
print("-" * 80)
print("A:", hardest["answer"])
print("gold:", repr(gold_answer(hardest)))

Q: There is one set of twins and one set of triplets. One twin is 7 years older than one triplet. If their combined ages are 44, how old is one of the twins?
--------------------------------------------------------------------------------
A: Let T = age of triplet
Twin's age = T + <<+7=7>>7
Twin's combined age = 2 * (T + 7)
Triplet's combined age = <<3=3>>3T
2 * (T + 7) + 3T = 44
2T + 14 + 3T = 44
5T + 14 = 44
5T = 30
T = <<6=6>>6
Twin = 6 + 7 = <<6+7=13>>13
The twins are each <<13=13>>13 years old.
#### 13
gold: '13'
